# Daily bars, up close

This notebook reads the DuckDB file written by `driftwood ingest`. If that file is missing it builds a two-year synthetic book, including a 2-for-1 split in `AAPL`, so the notebook still runs with no network and no API key.

Launch it from the repository root after `pip install -e ".[notebook]"`.

What a row is: one stock, one session. **Open** is the first print, **high** and **low** are the extremes, **close** is the last print, and **volume** is the share count. **Adjusted close** rewrites that close so splits and dividends do not look like economic gains or losses.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
root = cwd if (cwd / "driftwood").is_dir() else cwd.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from driftwood.config import database_path, load_env
from driftwood.data.ingest import ingest_synthetic
from driftwood.data.store import MarketStore

load_env()
db_path = database_path()
if not db_path.exists():
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with MarketStore(db_path) as store:
        result = ingest_synthetic(
            store,
            None,
            periods=504,
            seed=7,
            split_symbol="AAPL",
        )
    print(result.summary())
print(f"database: {db_path.resolve()}")

## Coverage

A name with far fewer rows than the others usually means the vendor returned nothing, or validation rejected the prints. It is not something to fill with zeros.


In [ ]:
with MarketStore(db_path) as store:
    coverage = store.coverage()
    sample = store.read_bars(symbols=["AAPL", "MSFT", "XOM", "JPM"])
    runs = store.read_runs()

coverage["start_date"] = pd.to_datetime(coverage["start_date"]).dt.date
coverage["end_date"] = pd.to_datetime(coverage["end_date"]).dt.date
print(f"{len(coverage)} symbols, {int(coverage['rows'].sum())} bars")
coverage.head(10)

In [ ]:
print(runs[["started_at", "source", "rows_written", "rows_rejected", "status"]].tail(5))
sample.groupby("symbol")[["close", "adj_close", "volume"]].tail(2)

## Adjusted close versus the raw print

On an ordinary day the two series move together. Across a split they should not: the raw close jumps, and the adjusted close should not show a crash. In the synthetic book the first half of `AAPL` has a ratio of 1, and the second half has a ratio of 2, because the raw price was halved and `adj_close` was left on the old scale.

A real yfinance file will not have that exact step. Dividends show up as a slowly changing ratio instead. If the ratio is exactly 1 for the whole history, either the name had no corporate actions in the window or the adjusted column was copied from the raw close because the vendor omitted it.


In [ ]:
apple = sample.loc[sample["symbol"] == "AAPL"].sort_values("date").copy()
apple["ratio"] = apple["adj_close"] / apple["close"]
apple["raw_return"] = apple["close"].pct_change()
apple["adj_return"] = apple["adj_close"].pct_change()
print(apple[["date", "close", "adj_close", "ratio", "raw_return", "adj_return"]].head(3))
print("ratio range:", round(apple["ratio"].min(), 4), "to", round(apple["ratio"].max(), 4))
# The split day, if this is the synthetic book, is the largest raw drop.
worst = apple["raw_return"].idxmin()
print(apple.loc[worst, ["date", "raw_return", "adj_return", "ratio"]].to_string())

## A first picture of the paths

Rebase each adjusted close to 1 on the first day of the sample. This is a picture of the series, not a backtest: there is no position, no cost, and no claim that the path was tradable in real time. Matplotlib comes from the `notebook` extra. Without it, the cell prints a table instead of drawing.


In [ ]:
wide = sample.pivot(index="date", columns="symbol", values="adj_close").sort_index()
rebased = wide / wide.iloc[0]
try:
    import matplotlib.pyplot as plt

    axis = rebased.plot(figsize=(9, 4), title="Adjusted close, rebased to 1")
    axis.set_ylabel("growth of 1")
    axis.set_xlabel("")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is not installed; showing the rebased table instead")
    rebased.tail()

## A leak, shown on purpose

A feature used at the close of day `t` may only depend on bars dated `t` or earlier. A trailing 21-day return satisfies that. A centered rolling mean does not: the value printed on day `t` uses prices from about ten days later.

The check is truncation. Recompute the feature on data cut off at `t`. The trailing return does not move. The centered mean does: once the cutoff removes the future sessions the window was using, that value can no longer be computed. The feature milestone turns this demonstration into a test that fails if a new column breaks the rule.


In [ ]:
close = apple.set_index("date")["adj_close"].sort_index()
if len(close) < 80:
    print(f"only {len(close)} AAPL rows; load a longer history to see the window")
else:
    cutoff = close.index[60]
    trailing = close.pct_change(21)
    centered = close.rolling(21, center=True).mean()
    truncated = close.loc[:cutoff]
    trailing_again = truncated.pct_change(21)
    centered_again = truncated.rolling(21, center=True).mean()
    print("cutoff", cutoff.date())
    print(
        "trailing return, full vs truncated:",
        float(trailing.loc[cutoff]),
        float(trailing_again.loc[cutoff]),
    )
    print(
        "centered mean, full vs truncated:",
        float(centered.loc[cutoff]),
        float(centered_again.loc[cutoff]),
    )

## Gaps and duplicate sessions

Daily research data is a business-day panel, not a guarantee that every name printed every day. This cell counts duplicate keys (there should be none after the primary key) and sessions that are missing relative to each name's own first and last date.


In [ ]:
with MarketStore(db_path) as store:
    bars = store.read_bars()

bars["date"] = pd.to_datetime(bars["date"])
duplicate_keys = int(bars.duplicated(["symbol", "date"]).sum())
print("duplicate symbol-date keys:", duplicate_keys)


def missing_sessions(group: pd.DataFrame) -> int:
    expected = pd.bdate_range(group["date"].min(), group["date"].max())
    return int(len(expected) - group["date"].nunique())


gaps = (
    bars.groupby("symbol", group_keys=False)
    .apply(missing_sessions, include_groups=False)
    .rename("missing_business_days")
    .sort_values(ascending=False)
)
gaps.head(8)